# Inference / corrector demo - Colab

Loads the trained checkpoint from Google Drive and runs one ayah through the word-level corrector. GPU is used when available.

In [ ]:
import os

os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# Run this clone line if the repo is not already uploaded to /content/model-asr-quran.
# !git clone https://github.com/<you>/model-asr-quran.git /content/model-asr-quran
%cd /content/model-asr-quran

!bash scripts/install_colab_deps.sh
!python -m pip install -q -e . --no-deps


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import requests
import torch

from quran_asr.alignment.corrector import load_corrector
from quran_asr.audio_io import load_audio

model_dir = Path("/content/drive/MyDrive/quran_asr/checkpoints")
assert (model_dir / "final").exists() or any(model_dir.glob("checkpoint-*")), f"No trained checkpoint found in {model_dir}"

audio_file = Path("/content/quran_asr/demo/001001.mp3")
audio_file.parent.mkdir(parents=True, exist_ok=True)
if not audio_file.exists():
    url = "https://everyayah.com/data/Husary_128kbps_Mujawwad/001001.mp3"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    audio_file.write_bytes(response.content)

device = "cuda" if torch.cuda.is_available() else "cpu"
corrector = load_corrector(model_dir, device=device)
audio, sr = load_audio(audio_file)
ref = "بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ"

print(f"device={device} audio={audio_file}")
for word in corrector.correct(audio, sr, ref):
    print(f"{word.index:>2} {word.status:<14} conf={word.confidence:.2f} {word.start:.2f}-{word.end:.2f}s  {word.text}")
